# W07: 카카오 알림 자동화

이벤트 알림 텍스트를 Gemini로 생성하고, Webhook이 있으면 전송하고 없으면 콘솔 출력합니다.


In [ ]:
import io
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for fp in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(fp)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}
    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        result = model.generate_content(prompt)
        return getattr(result, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

import requests

uploaded = safe_upload()
if uploaded:
    fn = list(uploaded.keys())[0]
    df = pd.read_csv(io.StringIO(uploaded[fn].decode("utf-8")))
else:
    df = pd.DataFrame(
        {
            "채널": ["카카오", "푸시", "알림톡"],
            "타겟": ["점심타임", "저녁타임", "새벽딜"],
            "할인율": [10, 20, 15],
            "혜택": ["배송비면제", "1+1", "사은품"],
        }
    )

if not set(["타겟", "할인율", "혜택"]).issubset(df.columns):
    raise ValueError("타겟, 할인율, 혜택 컬럼 필요")

webhook = os.getenv("KAKAO_WEBHOOK_URL", "")

messages = []
for r in df.itertuples():
    prompt = f"[{r.타겟}] 할인율 {r.할인율}%, 혜택 {r.혜택}. 고객 알림문구 2문장으로 작성"
    msg = use_gemini(prompt, f"[{r.타겟}] 이벤트 진행 중! {r.할인율}% 할인 + {r.혜택} 혜택이 적용됩니다.")
    messages.append(msg)

for msg in messages:
    if webhook:
        try:
            requests.post(webhook, json={"text": msg}, timeout=10)
            print("전송 완료")
        except Exception as e:
            print(f"전송 실패: {e}")
            print(msg)
    else:
        print(msg)

print("총", len(messages), "건 처리")
